# Generating the kraken2 + bracken database with the rumen catelogogue spike in.

Prerequisites:
- struo2 generated r232 gtdb kraken database
- Magnify rumen MAG catalogue reclassified by the r232 toolkit (toolkit needs to match the db version used to generate original kraken db).

## why do this?


In [ ]:
#Imports 
import pandas as pd

# Files:
Mgnify_dir = "../../r232/rumen_catalogue/mgnify_rumen_232/"


## bac120:

bac_class_summary = pd.read_csv(Mgnify_dir + "classify/gtdbtk.bac120.summary.tsv", sep="\t")
bac_markers_summary = pd.read_csv(Mgnify_dir + "identify/gtdbtk.bac120.markers_summary.tsv", sep="\t")


#archea

arc_class_summary = pd.read_csv(Mgnify_dir + "classify/gtdbtk.ar53.summary.tsv", sep="\t")
#arc_markers_summary = pd.read_csv(Mgnify_dir + "identify/gtdbtk.ar53.markers_summary.tsv", sep="\t")

# base metadata

base_meta = pd.read_csv("../../r232/filtered_metadata_with_taxids.tsv", sep="\t")

## Exploring the files

In [ ]:
bac_class_summary

In [ ]:
print(bac_markers_summary.columns)
bac_markers_summary

What does Struo2 need?
```
samples_col: 'ncbi_organism_name' # 
accession_col: 'accession' # Use the MAG ID
fasta_file_path_col: 'fasta_file_path' 
taxID_col: 'gtdb_taxid' 
taxonomy_col: 'gtdb_taxonomy' already in this table.
```

In [ ]:
# Start with what we have

sample_table = (
    pd
    .concat([bac_class_summary, arc_class_summary],axis=0) # mere the two dataframes together
    .loc[:,["user_genome", "classification", "closest_genome_reference"]] # select the columns we want to keep
    .rename(columns={"user_genome":"accession"}) # rename the column for what Struo2 expects
)
tax_levels = ["domain", "phylum", "class", "order", "family", "genus", "species"] 
sample_table[tax_levels] = (
    sample_table["classification"]
    .str.split(";", expand=True)
    .apply(lambda s: s.str.replace(r"^[a-z]__", "", regex=True))
)


sample_table

In [ ]:
#slicing the metadata to only include the columns we want to keep

base_meta = (
    base_meta
    .loc[:,["accession", "gtdb_taxonomy", "ncbi_organism_name", "gtdb_taxid"]]
    .rename(columns={"accession":"rep_accession"})
)
base_meta[tax_levels] = (
    base_meta["gtdb_taxonomy"]
    .str.split(";", expand=True)
    .apply(lambda s: s.str.replace(r"^[a-z]__", "", regex=True))
)

base_meta

In [ ]:
base_meta.count()

In [ ]:
# Try joining on the taxonomy.

sample_table = (
    sample_table
    .merge(base_meta, how="left", left_on="classification", 
           right_on="gtdb_taxonomy")
)
sample_table

In [ ]:
sample_table.count()